# Error Handling and File Safety

## Handling File I/O Errors Gracefully

In [1]:
import os

# Globomantics – Regional Configuration Loader
# Define expected config path (simulate a missing file)
region = "EMEA"
config_path = f"configs/{region}_config.ini"
default_config = {
    "currency": "USD",
    "timezone": "UTC",
    "retry_limit": "3"
}

# --- Step 1: Attempt to load configuration file ---
print(f"Loading config for region: {region}")

config_data = ""

try:
    config_file = open(config_path, 'r')
    config_data = config_file.read()
    print(f"Loaded config from file:\n{config_data}")
except FileNotFoundError:
    print(f"File not found: {config_path}. Creating file with default config...")
    config_data = "\n".join(f"{k}={v}" for k, v in default_config.items())
    
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    with open(config_path, 'w') as new_file:
        new_file.write(config_data)
    print(f"Created new config  file: {config_path}")
except PermissionError:
    print(f"Permission denied: Cannot access {config_path}. Using in-memory defaults.")
    config_data = "\n".join(f"{k}={v}" for k, v in default_config.items())
finally:
    if 'config_file'  in locals() and not config_file.closed:
        config_file.close()
        print("File closed correctly.")

Loading config for region: EMEA
Loaded config from file:
currency=USD
timezone=UTC
retry_limit=3
File closed correctly.


## Ensuring Safe File Access with Locking

In [2]:
import sys, time, platform

# Globomantics – Safe Concurrent Logger
log_file = "logs/shipment_activity.log"
os.makedirs("logs", exist_ok=True)

log_entry = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Shipment update from service {os.getpid()}\n"

def write_with_native_locking(filepath, data):
    if platform.system() == "Windows":
        import msvcrt
        with open(filepath, "a") as f:
            print("Locking with msvcrt...")
            msvcrt.locking(f.fileno(), msvcrt.LK_LOCK, len(data))
            f.write(data)
            f.flush()
            time.sleep(1)
            input("...")
            msvcrt.locking(f.fileno(), msvcrt.LK_UNLCK, len(data))
            print("Unlocked")
    else:
        import fcntl
        with open(filepath, "a") as f:
            print("Locking with  fcntl...")
            f.write(data)
            f.flush()
            time.sleep(1)
            input("From Linux...")
            fcntl.flock(f, fcntl.LOCK_UN)
            print("Unlocked...")

def write_with_portalocker(filepath, data):
    import portalocker
    with open(filepath, "a") as f:
        print("Locking with portalocker...")
        portalocker.lock(f, portalocker.LOCK_EX)
        f.write(data)
        f.flush()
        time.sleep(1)
        input("...")
        portalocker.unlock(f)
        print("Unlocked portalocker...")

print("\nAppending log entry safely...")
method = sys.argv[1] if len(sys.argv) > 1 else "native"

try:
    if method == "portlocker":
        write_with_portalocker(log_file, log_entry)
    else:
        write_with_native_locking(log_file, log_entry)
except Exception as e:
    print(f"Error during write {e}")

print("\nLog entry complete.")


Appending log entry safely...
Locking with  fcntl...


From Linux... y


Unlocked...

Log entry complete.
